# embench — end-to-end walkthrough

Benchmark embedding models on **your own** data — choose the right model for *your* dataset, not the leaderboard's.

This notebook covers:

1. **Setup**
2. **Offline benchmark** (runs immediately — no API keys, no downloads)
3. **Reading the results** (tables, ranking, aggregate score)
4. **Significance testing** — is one model *really* better, or is it noise?
5. **Real embedding models** (OpenAI / local / others)
6. **Ingest your own PDFs** → corpus → auto-generated questions → benchmark

## 1. Setup

Install the package (uncomment the line). The base install includes the API embedding backends **and** the PDF-ingestion stack (docling/pymupdf). Add `[local]` for local sentence-transformers models.

In [1]:
# %pip install embench           # core + API backends + PDF ingestion
# %pip install embench[local]    # + local sentence-transformers (PyTorch)

In [17]:
import os
import embench as eb

print("embench", eb.__version__)

# Find .env by walking up from the current directory, so this works whether the
# Jupyter kernel is rooted at the repo root or in examples/. (Or skip this
# entirely and set the keys as OS environment variables.)
_d = os.getcwd()
while True:
    _cand = os.path.join(_d, ".env")
    if os.path.exists(_cand):
        eb.load_env(_cand)
        print("loaded .env from", _cand)
        break
    _parent = os.path.dirname(_d)
    if _parent == _d:
        print("No .env found — set API keys in your OS environment instead.")
        break
    _d = _parent

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

embench 0.1.0
loaded .env from d:\embench\.env
OPENAI_API_KEY set: True


## 2. Offline benchmark (no keys needed)

`DummyModel` is a deterministic, hash-based baseline — no network, no keys. We'll build a tiny retrieval dataset inline and benchmark two dummy models against it. (Swap in real models in section 5.)

A `RetrievalDataset` is three dicts:
- `queries`: `{query_id: text}`
- `corpus`: `{doc_id: text}` — the documents to search over
- `qrels`: `{query_id: {doc_id: relevance}}` — which docs are relevant (the ground truth)

In [9]:
# A small but real retrieval set: each query is answered by exactly one doc,
# plus a couple of distractor docs that match nothing.
topics = {
    "q0": ("how do I reset my password",        "click forgot password on the login page"),
    "q1": ("which payment methods are accepted", "we accept visa, mastercard and paypal"),
    "q2": ("how do I cancel my subscription",    "cancel anytime under account settings billing"),
    "q3": ("is there a mobile app",              "our ios and android apps are on the app stores"),
    "q4": ("how long does shipping take",        "standard shipping arrives in three to five days"),
    "q5": ("can I get a refund",                 "refunds are issued within fourteen days of purchase"),
    "q6": ("do you offer student discounts",     "students get twenty percent off with a valid id"),
    "q7": ("how do I contact support",           "reach support by email or live chat anytime"),
}

queries = {qid: q for qid, (q, _) in topics.items()}
corpus = {f"d{i}": doc for i, (_, doc) in enumerate(topics.values())}
corpus["dx1"] = "the quarterly engineering roadmap and planning notes"
corpus["dx2"] = "office holiday schedule and parking information"
qrels = {f"q{i}": {f"d{i}": 1} for i in range(len(topics))}

retrieval = eb.RetrievalDataset(queries, corpus, qrels, name="mini-faq")
print(len(retrieval.queries), "queries,", len(retrieval.corpus), "docs")

8 queries, 10 docs


In [10]:
# Wrap each model in CachedModel so re-running cells doesn't recompute embeddings.
models = [
    eb.CachedModel(eb.DummyModel(dim=256)),
    eb.CachedModel(eb.DummyModel(dim=64)),
]

bench = eb.Benchmark(models, tasks=[eb.RetrievalTask(retrieval, k_values=[1, 3, 5])])
results = bench.run()

print("\n=== Quality ===")
print(results.to_table())

Running dummy-hash-256 | retrieval:mini-faq ...
  done in 0.00s (encoded 0 texts in 0.00s)
Running dummy-hash-64 | retrieval:mini-faq ...
  done in 0.00s (encoded 0 texts in 0.00s)

=== Quality ===
                retrieval:mini-faq/map@1  retrieval:mini-faq/map@3  retrieval:mini-faq/map@5  retrieval:mini-faq/mrr@1  retrieval:mini-faq/mrr@3  retrieval:mini-faq/mrr@5  retrieval:mini-faq/ndcg@1  retrieval:mini-faq/ndcg@3  retrieval:mini-faq/ndcg@5  retrieval:mini-faq/precision@1  retrieval:mini-faq/precision@3  retrieval:mini-faq/precision@5  retrieval:mini-faq/recall@1  retrieval:mini-faq/recall@3  retrieval:mini-faq/recall@5
model                                                                                                                                                                                                                                                                                                                                                                          

## 3. Reading the results

`results` is queryable several ways. Everything below is a pandas DataFrame or a plain Python object you can keep working with.

In [11]:
results.to_dataframe()          # wide: one row per model, one column per metric

,retrieval:mini-faq/map@1,retrieval:mini-faq/map@3,retrieval:mini-faq/map@5,retrieval:mini-faq/mrr@1,retrieval:mini-faq/mrr@3,retrieval:mini-faq/mrr@5,retrieval:mini-faq/ndcg@1,retrieval:mini-faq/ndcg@3,retrieval:mini-faq/ndcg@5,retrieval:mini-faq/precision@1,retrieval:mini-faq/precision@3,retrieval:mini-faq/precision@5,retrieval:mini-faq/recall@1,retrieval:mini-faq/recall@3,retrieval:mini-faq/recall@5
model,,,,,,,,,,,,,,,
dummy-hash-256,0.125,0.416667,0.416667,0.125,0.416667,0.416667,0.125,0.502965,0.502965,0.125,0.250000,0.150,0.125,0.75,0.750
dummy-hash-64,0.250,0.375000,0.456250,0.250,0.375000,0.456250,0.250,0.407732,0.558280,0.250,0.166667,0.175,0.250,0.50,0.875


In [12]:
print("best on ndcg@5:", results.best_model("ndcg@5"))
print("\nranking by ndcg@5:", results.ranking("ndcg@5"))

# One overall score per model = mean across all quality metrics.
print("\noverall ranking (mean of quality metrics):")
for i, (model, score) in enumerate(results.aggregate_ranking(), 1):
    print(f"  {i}. {model}: {score:.4f}")

# Speed / cost view (texts encoded, throughput; API $ cost when applicable).
print("\n=== Performance ===")
results.performance()

best on ndcg@5: dummy-hash-64

ranking by ndcg@5: [('dummy-hash-64', 0.558280209960674), ('dummy-hash-256', 0.5029648767857288)]

overall ranking (mean of quality metrics):
  1. dummy-hash-64: 0.3730
  2. dummy-hash-256: 0.3465

=== Performance ===


,encode_seconds,eval_seconds,texts_encoded,texts_per_sec
model,,,,
dummy-hash-256,0.0,0.003281,0.0,0.0
dummy-hash-64,0.0,0.003090,0.0,0.0


In [13]:
# Each metric also carries a per-query spread; show cells as 'mean +/- std'.
print(results.to_table(std=True))

               retrieval:mini-faq/map@1 retrieval:mini-faq/map@3 retrieval:mini-faq/map@5 retrieval:mini-faq/mrr@1 retrieval:mini-faq/mrr@3 retrieval:mini-faq/mrr@5 retrieval:mini-faq/ndcg@1 retrieval:mini-faq/ndcg@3 retrieval:mini-faq/ndcg@5 retrieval:mini-faq/precision@1 retrieval:mini-faq/precision@3 retrieval:mini-faq/precision@5 retrieval:mini-faq/recall@1 retrieval:mini-faq/recall@3 retrieval:mini-faq/recall@5
model                                                                                                                                                                                                                                                                                                                                                                                                                              
dummy-hash-256          0.1250 ± 0.3307          0.4167 ± 0.3005          0.4167 ± 0.3005          0.1250 ± 0.3307          0.4167 ± 0.3005          0.4167 ± 0.

## 4. Is the difference real? (significance testing)

A higher number isn't always a real win — it can be noise on these particular queries. embench runs a **paired randomization (Fisher) test** on the per-query scores to tell a genuine difference from chance.

Significance needs enough queries to have statistical power, so here we build a larger **synthetic** retrieval set (60 queries) and compare a sensible model against a deliberately useless one (a constant vector → chance retrieval). With enough queries, the good model now *significantly* wins. (On a tiny set like section 2's, the same gap can correctly read as a *tie* — too little evidence.)

In [14]:
import numpy as np


class ConstantModel(eb.BaseEmbeddingModel):
    """A useless baseline: identical vector for every text (chance retrieval)."""

    def __init__(self):
        super().__init__("constant")

    def _encode(self, texts):
        return np.ones((len(texts), 16), dtype=np.float32)


# Larger synthetic set so the test has enough queries to be conclusive:
# each query shares a unique term with exactly one doc.
N = 60
big_q = {f"q{i}": f"term{i} alpha beta gamma" for i in range(N)}
big_corpus = {f"d{i}": f"term{i} alpha beta gamma" for i in range(N)}
big_corpus["noise"] = "completely unrelated filler content"
big_qrels = {f"q{i}": {f"d{i}": 1} for i in range(N)}
big = eb.RetrievalDataset(big_q, big_corpus, big_qrels, name="synthetic")

results2 = eb.Benchmark(
    [eb.DummyModel(dim=256), ConstantModel()],
    tasks=[eb.RetrievalTask(big, k_values=[1, 5])],
).run(verbose=False)

# Pairwise: mean difference + p-value for every model pair.
results2.significance("ndcg@5")

,model_a,model_b,mean_a,mean_b,delta,p_value,significant
0,constant,dummy-hash-256,0.049141,0.987698,-0.938557,0.0001,True


In [15]:
# Per-model record: a 'win' is a significantly higher mean (p < 0.05).
results2.win_tie_loss("ndcg@5")

,score,wins,ties,losses
model,,,,
dummy-hash-256,0.987698,1,0,0
constant,0.049141,0,0,1


## 5. Benchmark real embedding models

Swap the dummy models for real ones. Hosted backends read their key from the environment (loaded by `eb.load_env()` above): `OPENAI_API_KEY`, `COHERE_API_KEY`, `VOYAGE_API_KEY`, `GOOGLE_API_KEY`, `HUGGINGFACE_API_KEY`.

```python
models = [
    eb.CachedModel(eb.OpenAIModel("text-embedding-3-small")),
    eb.CachedModel(eb.OpenAIModel("text-embedding-3-large")),
    eb.CachedModel(eb.SentenceTransformerModel("all-MiniLM-L6-v2")),  # needs embench[local]
    eb.CachedModel(eb.DummyModel(dim=256)),                            # baseline floor
]
```

The cell below tries OpenAI vs the dummy baseline; if no key is set it falls back to dummies so the notebook still runs.

In [16]:
import os

if os.environ.get("OPENAI_API_KEY"):
    real_models = [
        eb.CachedModel(eb.OpenAIModel("text-embedding-3-small")),
        eb.CachedModel(eb.DummyModel(dim=256)),  # baseline to beat
    ]
else:
    print("No OPENAI_API_KEY found — using dummy models so the notebook still runs.")
    real_models = [eb.CachedModel(eb.DummyModel(dim=256)), eb.CachedModel(eb.DummyModel(dim=64))]

res = eb.Benchmark(real_models, tasks=[eb.RetrievalTask(retrieval, k_values=[1, 5, 10])]).run()
print(res.to_table())

ImportError: openai is required for OpenAIModel. It ships with embench, so reinstall it with: pip install embench

## 6. Ingest your own PDFs

Turn real documents into a benchmark, in stages (each step persists its output so you never redo expensive work):

```
PDF(s) --extract--> text --chunk--> corpus --generate_queries--> dataset --> benchmark
```

Set `PDF_SOURCE` to **a single PDF file or a folder of PDFs** — the path is used directly, nothing is copied. Outputs land under `./data/`.

In [ ]:
# >>> EDIT THIS: a file OR a folder. Both work. <<<
PDF_SOURCE = r"C:\path\to\your.pdf"      # e.g. r"C:\Users\me\docs\" for a folder

import os
os.makedirs("data", exist_ok=True)

# Stage 1 - extract text. 'docling' (default) runs OCR, so scanned PDFs work too.
# Use method='pymupdf' for a fast text-layer-only extract of born-digital PDFs.
docs = eb.extract_text(
    PDF_SOURCE,
    method="docling",
    out_dir="data/extracted_text",   # persisted so you can re-chunk without re-extracting
)
print(f"extracted {len(docs)} document(s)")

In [ ]:
# Stage 2 - chunk into a {chunk_id: text} corpus (rule-based, deterministic).
corpus = eb.chunk_text(
    docs,                       # or "data/extracted_text" to load from disk
    method="recursive",         # 'recursive' (default) | 'sentence' | 'fixed'
    max_chars=1000,
    out_path="data/corpus.json",
)
print(f"{len(corpus)} chunks")

In [ ]:
# Stage 3 - auto-generate the supervision: an LLM writes the questions each
# chunk answers, and marks that chunk as the relevant answer (qrels).
# Needs OPENAI_API_KEY (default) or set method='google' for Gemini.
dataset = eb.generate_queries(
    corpus,                          # or "data/corpus.json"
    method="openai",
    n_queries=1,                     # questions per chunk
    max_chunks=None,                 # set e.g. 50 to cap cost on big docs
    out_path="data/dataset.json",    # a full RetrievalDataset, ready to load
)
print(f"{len(dataset['queries'])} queries over {len(dataset['corpus'])} chunks")

In [ ]:
# Stage 4 - benchmark embedding models on YOUR data.
my_data = eb.RetrievalDataset.from_json("data/dataset.json")

my_models = [
    eb.CachedModel(eb.OpenAIModel("text-embedding-3-small")),
    eb.CachedModel(eb.DummyModel(dim=256)),  # baseline floor
]

my_results = eb.Benchmark(
    my_models, tasks=[eb.RetrievalTask(my_data, k_values=[1, 5, 10])]
).run()

print(my_results.to_table())
print("\noverall ranking:", my_results.aggregate_ranking())
print("\nsignificance on ndcg@10:")
print(my_results.win_tie_loss("ndcg@10"))

my_results.to_csv("data/results.csv")
print("\nWrote data/results.csv")

### Notes

- **One-shot ingest:** `eb.pdf_to_corpus(PDF_SOURCE)` does extract + chunk in a single call (returns the corpus); then run `generate_queries` on it.
- **First docling run** downloads OCR/layout model weights (one-time, slow); later runs are fast.
- **Caching:** `CachedModel` stores embeddings on disk (`.embench_cache/`), so re-runs and added models don't re-pay for what's already computed.
- **Bring your own question generator:** `generate_queries(corpus, generator=lambda text, n: [...])` plugs in any model (or runs fully offline).